# MLflow for Deep Learning with Hyperopt

This notebook is written as a step-by-step note for an ML engineer learning how to use MLflow in a deep learning workflow.

By the end of the notebook, you will know how to:

- track deep learning experiments with MLflow
- organize Hyperopt trials as nested MLflow runs
- compare validation scores across trials
- retrain the best configuration and log the final model
- load the logged model back for inference
- optionally register the final model in the Model Registry


## Learning roadmap

We will move through the workflow in a practical order:

1. import the libraries and configure MLflow
2. load a tabular dataset for a regression problem
3. split the data into train, validation, and test sets
4. build a small Keras regression model
5. define a Hyperopt search space
6. track each trial as a nested MLflow run
7. retrain the best configuration and log the final model
8. load the model from MLflow and make predictions

This is a good mental model for production work as well: tune on validation data, keep the test set untouched until the very end, and use MLflow to preserve the full experiment history.


In [ ]:
## Import libraries
from pathlib import Path

import keras
import numpy as np
import pandas as pd
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.models import infer_signature

## Declare constants
SEED = 42
SEARCH_EPOCHS = 10
FINAL_EPOCHS = 20
MAX_EVALS = 6
EXPERIMENT_NAME = "dl-wine-quality-hyperopt-notes"

## Set random seed for keras
keras.utils.set_random_seed(SEED)

## Decide project root
project_root = Path.cwd()
if project_root.name == "03-DL_MLFLOW":
    project_root = project_root.parent

tracking_db = project_root / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{tracking_db.as_posix()}")

print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment name: {EXPERIMENT_NAME}")


e:\AI\MlOPS\MLFLOW\mlops_venv\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
e:\AI\MlOPS\MLFLOW\mlops_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tracking URI: sqlite:///e:/AI/MlOPS/MLFLOW/mlflow.db
Experiment name: dl-wine-quality-hyperopt-notes


## Step 1: Load the dataset

We will use the white wine quality dataset. The target column is `quality`, so this becomes a regression problem.

For this notebook, the data source is remote so we can focus on MLflow and not on local data management.


In [2]:
data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",
)

data.head()


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


## Step 2: Create train, validation, and test splits

A clean split strategy matters when tuning models:

- the training set fits the model weights
- the validation set is used by Hyperopt to compare configurations
- the test set is kept untouched until the best configuration is chosen

This separation helps us avoid optimistic evaluation.


In [ ]:
## Set feature and target columns
TARGET_COLUMN = "quality"
FEATURE_COLUMNS = [column for column in data.columns if column != TARGET_COLUMN]

## Split data into train and test and validation
train_df, test_df = train_test_split(data, test_size=0.20, random_state=SEED)
train_df, valid_df = train_test_split(train_df, test_size=0.20, random_state=SEED)

X_train = train_df[FEATURE_COLUMNS].copy()
y_train = train_df[TARGET_COLUMN].copy()

X_valid = valid_df[FEATURE_COLUMNS].copy()
y_valid = valid_df[TARGET_COLUMN].copy()

X_test = test_df[FEATURE_COLUMNS].copy()
y_test = test_df[TARGET_COLUMN].copy()

pd.DataFrame(
    {"rows": [len(X_train), len(X_valid), len(X_test)]},
    index=["train", "validation", "test"],
)


,rows
train,3134
validation,784
test,980


## Step 3: Build reusable training helpers

The model is intentionally small so the learning focus stays on experiment tracking.

A few useful ideas to notice here:

- input normalization is part of the Keras model
- Hyperopt samples parameters, but we sanitize them before training
- the helper returns both the trained model and the metrics we want to log in MLflow


In [ ]:
## Function to convert parameter's values into float
def sanitize_params(params):
    return {
        "learning_rate": float(params["learning_rate"]),
        "momentum": float(params["momentum"]),
        "hidden_units": int(params["hidden_units"]),
    }


## Function to build the model (Architecture + Compile)
def build_model(params, feature_count, normalization_data):
    normalizer = keras.layers.Normalization()
    normalizer.adapt(normalization_data)

    model = keras.Sequential(
        [
            keras.Input(shape=(feature_count,)),
            normalizer,
            keras.layers.Dense(params["hidden_units"], activation="relu"),
            keras.layers.Dense(1),
        ]
    )

    optimizer = keras.optimizers.SGD(
        learning_rate=params["learning_rate"],
        momentum=params["momentum"],
    )

    model.compile(
        optimizer=optimizer,
        loss="mean_squared_error",
        metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
    )
    return model


## Function to train and evaluate the model
def train_and_evaluate(params, X_train, y_train, X_valid, y_valid, epochs, batch_size=64):
    model = build_model(params, X_train.shape[1], X_train.to_numpy())

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_valid, y_valid),
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
    )

    valid_loss, valid_rmse = model.evaluate(X_valid, y_valid, verbose=0)

    metrics = {
        "val_loss": float(valid_loss),
        "val_rmse": float(valid_rmse),
        "train_rmse_last_epoch": float(history.history["rmse"][-1]),
        "val_rmse_last_epoch": float(history.history["val_rmse"][-1]),
    }
    return model, history, metrics


## Step 4: Define the Hyperopt search space and the MLflow objective

This is the core learning step.

Each Hyperopt trial is wrapped inside `mlflow.start_run(nested=True)`, which means:

- the full tuning session can live inside one parent run
- every trial gets its own child run
- the MLflow UI becomes much easier to read when you compare runs

The objective function returns validation RMSE because that is the metric Hyperopt should minimize.


In [ ]:
## Define search space (set of hyperparameter values)
search_space = {
    "learning_rate": hp.loguniform("learning_rate", np.log(1e-4), np.log(5e-2)),
    "momentum": hp.uniform("momentum", 0.0, 0.95),
    "hidden_units": hp.quniform("hidden_units", 32, 128, 32),
}


def objective(params):
    params = sanitize_params(params)

    with mlflow.start_run(nested=True, run_name="hyperopt-trial") as trial_run:
        model, history, metrics = train_and_evaluate(
            params,
            X_train,
            y_train,
            X_valid,
            y_valid,
            epochs=SEARCH_EPOCHS,
        )

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.log_dict(history.history, "trial_history.json")

        return {
            "loss": metrics["val_rmse"],
            "status": STATUS_OK,
            "run_id": trial_run.info.run_id,
            "params": params,
        }


## Step 5: Run Hyperopt and log the final best model

The parent run captures the overall tuning session.

Inside that parent run we will:

- run Hyperopt for several trials
- find the best hyperparameter set based on validation RMSE
- retrain using the winning configuration
- evaluate once on the test set
- log the final model artifact to MLflow

This keeps the search history separate from the final promoted model.


In [7]:
mlflow.set_experiment(EXPERIMENT_NAME)
trials = Trials()

with mlflow.start_run(run_name="hyperopt-search-parent") as parent_run:
    mlflow.set_tag("notebook_role", "step-by-step notes")
    mlflow.log_params(
        {
            "search_epochs": SEARCH_EPOCHS,
            "final_epochs": FINAL_EPOCHS,
            "max_evals": MAX_EVALS,
            "target_column": TARGET_COLUMN,
        }
    )

    best_raw_params = fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=MAX_EVALS,
        trials=trials,
        rstate=np.random.default_rng(SEED),
    )

    best_params = sanitize_params(best_raw_params)
    best_trial = min(trials.results, key=lambda result: result["loss"])

    final_model, final_history, final_valid_metrics = train_and_evaluate(
        best_params,
        X_train,
        y_train,
        X_valid,
        y_valid,
        epochs=FINAL_EPOCHS,
    )

    test_loss, test_rmse = final_model.evaluate(X_test, y_test, verbose=0)
    input_example = X_train.head(3).to_numpy(dtype=np.float32)
    predictions_for_signature = final_model.predict(input_example, verbose=0)
    signature = infer_signature(input_example, predictions_for_signature)

    mlflow.log_params({f"best_{key}": value for key, value in best_params.items()})
    mlflow.log_metric("best_search_val_rmse", float(best_trial["loss"]))
    mlflow.log_metric("final_val_rmse", final_valid_metrics["val_rmse"])
    mlflow.log_metric("final_test_rmse", float(test_rmse))
    mlflow.log_dict(final_history.history, "final_training_history.json")
    mlflow.tensorflow.log_model(
        final_model,
        "model",
        signature=signature,
        input_example=input_example,
    )

    parent_run_id = parent_run.info.run_id
    best_trial_run_id = best_trial["run_id"]
    best_model_uri = f"runs:/{parent_run_id}/model"

print(f"Parent run ID: {parent_run_id}")
print(f"Best child run ID: {best_trial_run_id}")
print(f"Best parameters: {best_params}")
print(f"Best model URI: {best_model_uri}")


100%|██████████| 6/6 [00:13<00:00,  2.18s/trial, best loss: 0.6927174925804138]


2026/03/31 21:43:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Parent run ID: 540f2addeeb74fd0a53cf0ba7454520f
Best child run ID: bd0dcfee2b5941b39afe94ed4c6c4f82
Best parameters: {'learning_rate': 0.00471718136105401, 'momentum': 0.7729726372582734, 'hidden_units': 128}
Best model URI: runs:/540f2addeeb74fd0a53cf0ba7454520f/model


## Step 6: Compare trial results inside the notebook

The MLflow UI is the best place to compare runs visually, but it is also helpful to summarize trial results directly in the notebook.


In [8]:
trial_results = pd.DataFrame(
    [
        {
            "trial_number": index + 1,
            "run_id": result["run_id"],
            "val_rmse": round(result["loss"], 4),
            "learning_rate": result["params"]["learning_rate"],
            "momentum": result["params"]["momentum"],
            "hidden_units": result["params"]["hidden_units"],
        }
        for index, result in enumerate(trials.results)
    ]
).sort_values("val_rmse").reset_index(drop=True)

trial_results


,trial_number,run_id,val_rmse,learning_rate,momentum,hidden_units
0,1,bd0dcfee2b5941b39afe94ed4c6c4f82,0.6927,0.004717,0.772973,128
1,2,dc648cd8fa774722b9a662b44ccd6b97,0.8663,0.004382,0.285400,128
2,6,7ecb66b039c147a1b01a3daa9cc6e54c,1.3282,0.000272,0.716991,64
3,4,a7874958be3b492a9d4c31f8dd649025,1.5426,0.000374,0.383589,96
4,5,8510b0e86f6246c2af440d29f151ca14,1.7620,0.000182,0.477759,64
5,3,4b9816300fd44195bb95ddc1136a0776,2.6684,0.000158,0.139615,96


To inspect the same runs in the MLflow UI, open a terminal at the project root and run:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Then open the experiment named `dl-wine-quality-hyperopt-notes`.


## Step 7: Load the logged model for inference

A very useful MLflow habit is to test model loading immediately after logging the artifact.

If the model loads and predicts correctly here, you already have more confidence before moving to serving or deployment.


In [ ]:
loaded_model = mlflow.pyfunc.load_model(best_model_uri)

sample_features = X_test.head(5).copy()
sample_predictions = np.squeeze(loaded_model.predict(sample_features))

pd.DataFrame(
    {
        "actual_quality": y_test.head(5).to_numpy(),
        "predicted_quality": np.round(sample_predictions, 3),
    }
)


## Step 8: Optional model registration

If you want to promote the best model into the MLflow Model Registry, keep the code below and switch the flag to `True`.

This is useful once you move from experimentation into a more formal model lifecycle.


In [ ]:
REGISTER_MODEL = False
REGISTERED_MODEL_NAME = "wine-quality-dl-hyperopt"

if REGISTER_MODEL:
    registration = mlflow.register_model(best_model_uri, REGISTERED_MODEL_NAME)
    print(f"Registered model version: {registration.version}")
else:
    print("Set REGISTER_MODEL = True when you are ready to register the best model.")


## Final notes

What this notebook teaches you in practice:

- MLflow can track each Hyperopt trial as a child run
- the parent run can represent the overall tuning job
- validation RMSE should drive hyperparameter search
- the test set should be used only after the best configuration is selected
- the final best model can be logged, loaded, and registered from the same workflow

A good next exercise is to increase `MAX_EVALS`, add more layers or dropout, and compare how the MLflow run history changes.
